# 01 - Data Preparation for RFT

This notebook presents the dataset prepared for Reinforcement Fine-Tuning (RFT) of a retail customer service **Planner**.

## Dataset Origin

The dataset is based on **tau-bench** ([Sierra Research](https://github.com/sierra-research/tau-bench)), a benchmark for evaluating LLM agents on realistic retail customer service tasks.

- **115 tasks** covering order management, returns, exchanges, and account modifications
- **15 tools** for retail operations
- Tasks simulate real customer interactions with various personalities and edge cases

> **Note**: The tau-bench data has been pre-processed and formatted for RFT training. The prepared datasets are provided in `data/datasets/`.

## Notebook Outline

1. **RFT Data Format** - Expected format for Azure OpenAI Fine-tuning API
2. **Available Tools** - The 15 retail tools
3. **Load Datasets** - Load train/val/test splits
4. **Validate Format** - Ensure correct structure
5. **EDA** - Exploratory data analysis

---
## 1. RFT Data Format

Azure OpenAI Reinforcement Fine-Tuning expects **JSONL** files with this structure:

```json
{
  "messages": [
    {"role": "developer", "content": "System prompt with tool descriptions..."},
    {"role": "user", "content": "Customer request to handle"}
  ],
  "reference_answer": {
    "expected_tools": ["find_user_id_by_name_zip", "get_user_details", "..."]
  }
}
```

### Key Points

| Field | Description |
|-------|-------------|
| `messages[0].role` | Use `"developer"` (not `"system"`) for instructions |
| `messages[1].role` | `"user"` contains the customer request |
| `reference_answer` | Native JSON object (not a string) |
| `expected_tools` | List of tools in execution order |

The **grader** during RFT training compares the model's predicted tools against `expected_tools` to compute an F1 score.

---
## 2. Available Tools

The retail customer service system has **15 tools** organized in 4 categories:

### Account Tools (4)
| Tool | Description |
|------|-------------|
| `find_user_id_by_email` | Find user ID by email address |
| `find_user_id_by_name_zip` | Find user ID by first name, last name, and zip code |
| `get_user_details` | Get user details including their orders |
| `modify_user_address` | Modify the default address of a user |

### Order Tools (7)
| Tool | Description |
|------|-------------|
| `get_order_details` | Get the status and details of an order |
| `cancel_pending_order` | Cancel a pending order |
| `modify_pending_order_address` | Modify the shipping address of a pending order |
| `modify_pending_order_items` | Modify items in a pending order |
| `modify_pending_order_payment` | Modify the payment method of a pending order |
| `get_product_details` | Get the inventory details of a product |
| `list_all_product_types` | List all available product types |

### Refund Tools (2)
| Tool | Description |
|------|-------------|
| `return_delivered_order_items` | Return items from a delivered order |
| `exchange_delivered_order_items` | Exchange items in a delivered order |

### Utility Tools (2)
| Tool | Description |
|------|-------------|
| `calculate` | Calculate the result of a mathematical expression |
| `transfer_to_human_agents` | Transfer to human agent with a summary |

---
## 3. Load Datasets

In [ ]:
import json
import sys
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent to path for src imports
sys.path.insert(0, str(Path.cwd().parent))

from src.settings import DATASETS_DIR, EDA_OUTPUTS_DIR

# Configure plotting
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print(f"Datasets directory: {DATASETS_DIR}")
print(f"EDA outputs: {EDA_OUTPUTS_DIR}")

In [ ]:
def load_jsonl(path: Path) -> list:
    """Load a JSONL file into a list of dictionaries."""
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

# Load datasets
train_data = load_jsonl(DATASETS_DIR / "train.jsonl")
val_data = load_jsonl(DATASETS_DIR / "val.jsonl")
test_data = load_jsonl(DATASETS_DIR / "test.jsonl")

print(f"Train: {len(train_data)} samples")
print(f"Val:   {len(val_data)} samples")
print(f"Test:  {len(test_data)} samples")
print(f"Total: {len(train_data) + len(val_data) + len(test_data)} samples")

### Example Sample

Let's examine a sample to see the actual data structure and content.

In [ ]:
# Show a sample
sample = train_data[0]

print("=" * 60)
print("SAMPLE STRUCTURE")
print("=" * 60)
print(f"\nKeys: {list(sample.keys())}")
print(f"\nMessages: {len(sample['messages'])} messages")
print(f"  - Developer: {len(sample['messages'][0]['content'])} chars")
print(f"  - User: {len(sample['messages'][1]['content'])} chars")
print(f"\nExpected tools: {sample['reference_answer']['expected_tools']}")

In [ ]:
# Show the user request
print("USER REQUEST:")
print("-" * 40)
print(sample['messages'][1]['content'])

---
## 4. Validate Format

Ensure all samples have the correct structure for RFT.

In [ ]:
def validate_sample(sample: dict, idx: int) -> list:
    """Validate a single sample. Returns list of errors."""
    errors = []
    
    # Check messages
    if 'messages' not in sample:
        errors.append(f"Sample {idx}: missing 'messages'")
    elif len(sample['messages']) != 2:
        errors.append(f"Sample {idx}: expected 2 messages, got {len(sample['messages'])}")
    else:
        if sample['messages'][0].get('role') != 'developer':
            errors.append(f"Sample {idx}: first message role should be 'developer'")
        if sample['messages'][1].get('role') != 'user':
            errors.append(f"Sample {idx}: second message role should be 'user'")
    
    # Check reference_answer
    if 'reference_answer' not in sample:
        errors.append(f"Sample {idx}: missing 'reference_answer'")
    elif 'expected_tools' not in sample['reference_answer']:
        errors.append(f"Sample {idx}: missing 'expected_tools' in reference_answer")
    
    return errors

def validate_dataset(data: list, name: str) -> bool:
    """Validate all samples in a dataset."""
    all_errors = []
    for idx, sample in enumerate(data):
        errors = validate_sample(sample, idx)
        all_errors.extend(errors)
    
    if all_errors:
        print(f"\n{name}: {len(all_errors)} errors found")
        for err in all_errors[:5]:  # Show first 5
            print(f"  - {err}")
        return False
    else:
        print(f"{name}: All {len(data)} samples valid")
        return True

# Validate all datasets
print("Validating datasets...\n")
all_valid = all([
    validate_dataset(train_data, "Train"),
    validate_dataset(val_data, "Val"),
    validate_dataset(test_data, "Test")
])

print(f"\nAll datasets valid: {all_valid}")

---
## 5. Exploratory Data Analysis

Let's analyze the dataset to understand its characteristics before training. We'll look at:
- Basic statistics (instruction length, tools per sample)
- Tool usage frequency
- Common patterns in tool sequences

In [ ]:
# Combine all data for analysis
all_data = train_data + val_data + test_data

def extract_features(sample: dict) -> dict:
    """Extract features from a sample for analysis."""
    instruction = sample['messages'][1]['content']
    tools = sample['reference_answer']['expected_tools']
    
    return {
        'instruction_length': len(instruction),
        'instruction_words': len(instruction.split()),
        'num_tools': len(tools),
        'tools': tools,
        'tool_sequence': ' -> '.join(tools),
        'first_tool': tools[0] if tools else None,
        'last_tool': tools[-1] if tools else None
    }

# Build DataFrame
df = pd.DataFrame([extract_features(s) for s in all_data])
print(f"Dataset shape: {df.shape}")
df.head()

### Dataset Statistics

Basic metrics about instruction complexity and the number of tools required per task.

In [ ]:
print("=" * 50)
print("DATASET STATISTICS")
print("=" * 50)

print(f"\nInstruction Length (chars):")
print(f"  Min: {df['instruction_length'].min()}")
print(f"  Max: {df['instruction_length'].max()}")
print(f"  Mean: {df['instruction_length'].mean():.1f}")

print(f"\nTools per Sample:")
print(f"  Min: {df['num_tools'].min()}")
print(f"  Max: {df['num_tools'].max()}")
print(f"  Mean: {df['num_tools'].mean():.1f}")
print(f"  Median: {df['num_tools'].median():.1f}")

### Distribution Visualizations

How are tools and instruction lengths distributed across samples? These distributions help understand the dataset's complexity range.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tools per sample distribution
ax1 = axes[0]
sns.histplot(df['num_tools'], bins=range(0, df['num_tools'].max() + 2), ax=ax1, color='steelblue')
ax1.axvline(df['num_tools'].mean(), color='red', linestyle='--', label=f'Mean: {df["num_tools"].mean():.1f}')
ax1.set_xlabel('Number of Tools')
ax1.set_ylabel('Count')
ax1.set_title('Distribution of Tools per Sample')
ax1.legend()

# Instruction word count distribution
ax2 = axes[1]
sns.histplot(df['instruction_words'], bins=20, ax=ax2, color='seagreen')
ax2.axvline(df['instruction_words'].mean(), color='red', linestyle='--', label=f'Mean: {df["instruction_words"].mean():.1f}')
ax2.set_xlabel('Word Count')
ax2.set_ylabel('Count')
ax2.set_title('Distribution of Instruction Length')
ax2.legend()

plt.tight_layout()
plt.savefig(EDA_OUTPUTS_DIR / 'eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {EDA_OUTPUTS_DIR / 'eda_distributions.png'}")

### Tool Usage Frequency

Which tools are used most often? This reveals the core operations in retail customer service.

In [ ]:
# Count tool occurrences
tool_counts = Counter()
for tools in df['tools']:
    tool_counts.update(tools)

# Create frequency DataFrame
tool_freq = pd.DataFrame([
    {'tool': tool, 'count': count, 'percentage': count / len(df) * 100}
    for tool, count in tool_counts.most_common()
])

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

colors = sns.color_palette('viridis', len(tool_freq))
bars = ax.barh(tool_freq['tool'], tool_freq['count'], color=colors)

ax.set_xlabel('Number of Samples')
ax.set_title('Tool Usage Frequency Across All Samples')
ax.invert_yaxis()  # Most frequent at top

# Add percentage labels
for bar, pct in zip(bars, tool_freq['percentage']):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{pct:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(EDA_OUTPUTS_DIR / 'eda_tool_usage.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {EDA_OUTPUTS_DIR / 'eda_tool_usage.png'}")

### First Tool Analysis

Which tools typically start a sequence? This reveals the entry points for customer service interactions.

In [ ]:
# First tool distribution
first_tool_counts = df['first_tool'].value_counts()

# Plot
fig, ax = plt.subplots(figsize=(10, 5))

colors = sns.color_palette('Set2', len(first_tool_counts))
bars = ax.barh(first_tool_counts.index, first_tool_counts.values, color=colors)

ax.set_xlabel('Number of Samples')
ax.set_title('First Tool in Sequence (Entry Points)')
ax.invert_yaxis()

# Add percentage labels
for bar, count in zip(bars, first_tool_counts.values):
    pct = count / len(df) * 100
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{pct:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\n~{(first_tool_counts.head(2).sum() / len(df) * 100):.0f}% of tasks start with user identification")

### Most Common Tool Sequences

What are the typical workflows? High sequence diversity indicates the model needs to learn flexible planning rather than memorizing fixed patterns.

In [ ]:
# Top sequences
sequence_counts = df['tool_sequence'].value_counts()
top_10 = sequence_counts.head(10)

# Shorten tool names for display
def shorten_sequence(seq):
    """Shorten tool names for readable labels."""
    replacements = {
        'find_user_id_by_name_zip': 'find_by_zip',
        'find_user_id_by_email': 'find_by_email',
        'get_user_details': 'user_details',
        'get_order_details': 'order_details',
        'get_product_details': 'product_details',
        'exchange_delivered_order_items': 'exchange',
        'return_delivered_order_items': 'return',
        'cancel_pending_order': 'cancel',
        'modify_pending_order_items': 'modify_items',
        'modify_pending_order_address': 'modify_addr',
        'modify_pending_order_payment': 'modify_pay',
        'transfer_to_human_agents': 'transfer',
        'list_all_product_types': 'list_products',
        'modify_user_address': 'modify_user_addr',
        'calculate': 'calc'
    }
    for old, new in replacements.items():
        seq = seq.replace(old, new)
    return seq

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

short_labels = [shorten_sequence(seq) for seq in top_10.index]
colors = sns.color_palette('coolwarm', len(top_10))
bars = ax.barh(range(len(top_10)), top_10.values, color=colors)

ax.set_yticks(range(len(top_10)))
ax.set_yticklabels(short_labels, fontsize=8)
ax.set_xlabel('Number of Samples')
ax.set_title('Top 10 Most Common Tool Sequences')
ax.invert_yaxis()

# Add count labels
for bar, count in zip(bars, top_10.values):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{count}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nTotal unique sequences: {len(sequence_counts)} out of {len(df)} samples ({len(sequence_counts)/len(df)*100:.0f}% diversity)")

---
## Summary

### Dataset Ready for RFT

| Split | Samples |
|-------|--------|
| Train | 85 |
| Val | 20 |
| Test | 10 |
| **Total** | **115** |

### Key Insights

1. **User identification is critical**: ~94% of tasks start with `find_user_id_by_*`
2. **Order details are universal**: `get_order_details` appears in nearly all tasks
3. **Diverse sequences**: 53% unique sequences shows task variety
4. **Reasonable complexity**: Average of 5 tools per task

### Next Step

Proceed to **Notebook 02 - Training** to fine-tune the planner model using RFT.